# Vector Database Selection

**Module:** 02 — Vector Databases

Decision frameworks for small apps, enterprise, cloud, and performance.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Select proportionate stacks
- Weigh cloud vs DIY
- Optimize with measurements


## Small Apps

**Definition.** Modest N/QPS; optimize for speed-to-value.

**Why it matters.** Overbuild kills velocity.

**How it works.** Chroma/Lance/pgvector/single-node Qdrant; portable ingest.

**Intuition.** Bicycle not freight train.

**Common pitfalls.**
- Premature distributed platforms

**When to use.** MVPs/internal tools.


In [ ]:
def rec(n):
    return 'faiss/numpy' if n<2e5 else 'chroma/qdrant' if n<2e6 else 'cluster/managed'
print([(n,rec(n)) for n in [1e4,5e5,2e7]])


In [ ]:
class Sink:
    def __init__(self): self.d={}
    def upsert(self,ids,V,P):
        for i,v,p in zip(ids,V,P): self.d[i]=(v,p); return len(ids)
print(Sink().upsert(['a'],[[0.1]],[{}]))


In [ ]:
print('side project infra > $50/mo → simplify')


### Try it yourself — Small Apps

1. Stack for 100k personal KB.


## Enterprise

**Definition.** Security, tenancy, SLAs, hybrid, operable failures.

**Why it matters.** Compliance > clever indexes.

**How it works.** Requirements matrix + realistic POC.

**Intuition.** SRE/security must sleep.

**Common pitfalls.**
- POC without ACL filters

**When to use.** Regulated/multi-team platforms.


In [ ]:
opts={'Qdrant':{'sec':4,'ops':3,'cost':4},'Pinecone':{'sec':4,'ops':5,'cost':3}}
w={'sec':0.4,'ops':0.3,'cost':0.3}
for n,s in opts.items(): print(n, sum(s[k]*w[k] for k in w))


In [ ]:
print(['metadata_filter','namespace','db_per_tenant'])


In [ ]:
for c in ['encryption','TLS','RBAC','backups']: print('[ ]',c)


### Try it yourself — Enterprise

1. Rubric for a bank internal RAG.


## Cloud

**Definition.** Managed DBs or self-host in your cloud.

**Why it matters.** Region/latency/compliance/cost.

**How it works.** Same-region app+DB+LLM when possible.

**Intuition.** Keep pins near the map.

**Common pitfalls.**
- Cross-region +100ms

**When to use.** Internet-facing products.


In [ ]:
parts={'embed':40,'search':30,'rerank':60,'llm':70}; print(sum(parts.values()), 'budget 200')


In [ ]:
print(len(set({'app':'eastus','db':'eastus','llm':'eastus'}.values()))==1)


In [ ]:
def tco(db,h,rate=150): return db+h*rate
print(tco(800,4), tco(300,40))


### Try it yourself — Cloud

1. Region topology sketch.


## Performance Optimization

**Definition.** Latency/throughput/recall/cost via knobs, cache, compression.

**Why it matters.** Missed SLOs or burned money.

**How it works.** Measure → gates → sweep → compress → cache → scale.

**Intuition.** Biggest bottleneck first.

**Common pitfalls.**
- Benchmarks without filters

**When to use.** When SLOs/costs hurt.

```mermaid
flowchart TD
 M[Measure]-->B[Bottleneck]
 B-->I[Tune index]; B-->E[Embed]; B-->R[Rerank]; B-->L[LLM]
```


In [ ]:
import itertools
best=None
for ef,rep in itertools.product([32,64,128],[1,2]):
    lat=10+ef*0.08+(2-rep)*5; rec=0.9+min(ef,128)/128*0.09; cand=(rec,-lat,ef,rep)
    best=cand if best is None or cand>best else best
print(best)


In [ ]:
import hashlib
print(hashlib.sha256(b'bge|acme|refund').hexdigest()[:16])


In [ ]:
def overfetch(k,sel,s=2.0): return int(k/max(sel,1e-6)*s)
print(overfetch(10,0.05))


In [ ]:
print((0.98>=0.97 and 45<=50), (0.99>=0.97 and 80<=50))


### Try it yourself — Performance Optimization

1. Ordered optimization backlog.


## Glossary

- **SLO**: Service level objective
- **TCO**: Total cost of ownership


## Operator Checklist

- [ ] model+dim+metric documented
- [ ] collection naming by env/model
- [ ] auth-bound tenant filters
- [ ] recall@k + p95 gates
- [ ] re-embed/delete playbooks


In [ ]:
for i,x in enumerate(['model+dim+metric','naming','tenant filter','recall gate','p95 gate','reembed playbook'],1):
    print(f'{i}. [ ] {x}')


### Try it yourself — Readiness

1. Fill the checklist for a real system.
2. Name the top 6-month risk if ignored.
3. Pick one paging metric.


In [ ]:
def gib(n,dim,b=4,r=2,o=1.5):
    return n*dim*b*r*o/(1024**3)
print(f'{gib(5_000_000,768):.1f} GiB for 5M×768-d')


## End-to-End Flow

```mermaid
flowchart LR
  D[Docs]-->C[Chunk]-->E[Embed]-->U[Upsert]-->I[(Index)]
  Q[Query]-->E2[Embed]-->S[Search+filter]
  I-->S-->A[App/RAG]
```


In [ ]:
def inject_tenant(auth, user=None):
    base={'tenant':{'$eq':auth}}
    return {'$and':[base,user]} if user else base
print(inject_tenant('acme',{'lang':{'$eq':'en'}}))


## Summary & Key Takeaways

- Match complexity to N/skills.
- Enterprise is security/ops constrained.
- Optimize recall+p95 together.

### Practice

One-page memo for 10M-vector SaaS FAQ stack.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
